In [2]:
import pandas as pd
import numpy as np

# Load both
kelly = pd.read_csv('datashare\\datashare.csv', low_memory=True)
crsp  = pd.read_csv('crsp_returns.csv', parse_dates=['date'])

# --- Parse Kelly date to match CRSP format ---
# Kelly DATE is YYYYMMDD integer, CRSP date is YYYY-MM-01
kelly['date'] = pd.to_datetime(
    kelly['DATE'].astype(str), format='%Y%m%d'
)
kelly['date'] = kelly['date'].dt.to_period('M').dt.to_timestamp()

print("Kelly date range:", kelly['date'].min(), "to", kelly['date'].max())
print("CRSP date range:",  crsp['date'].min(),  "to", crsp['date'].max())
print("Kelly rows:", len(kelly))
print("CRSP rows:",  len(crsp))

# --- Check permno overlap ---
kelly_permnos = set(kelly['permno'].unique())
crsp_permnos  = set(crsp['permno'].unique())

print(f"\nKelly unique permnos: {len(kelly_permnos)}")
print(f"CRSP unique permnos:  {len(crsp_permnos)}")
print(f"In both:              {len(kelly_permnos & crsp_permnos)}")
print(f"In Kelly but not CRSP: {len(kelly_permnos - crsp_permnos)}")


Kelly date range: 1957-01-01 00:00:00 to 2021-12-01 00:00:00
CRSP date range: 1957-01-01 00:00:00 to 2024-11-01 00:00:00
Kelly rows: 4117300
CRSP rows: 4229810

Kelly unique permnos: 32793
CRSP unique permnos:  32365
In both:              32365
In Kelly but not CRSP: 428


In [3]:

# --- Merge ---
# Keep only ret and me from CRSP (everything else comes from Kelly)
crsp_to_merge = crsp[['permno', 'date', 'ret', 'ret_raw', 'me']].copy()

kelly_with_ret = kelly.merge(
    crsp_to_merge,
    on=['permno', 'date'],
    how='left'
)

print(f"\nAfter merge:")
print(f"  Rows: {len(kelly_with_ret)}")
print(f"  Rows with ret: {kelly_with_ret['ret'].notna().sum():,}")
print(f"  Rows missing ret: {kelly_with_ret['ret'].isna().sum():,}")
print(f"  Match rate: {kelly_with_ret['ret'].notna().mean():.2%}")



After merge:
  Rows: 4117300
  Rows with ret: 3,895,198
  Rows missing ret: 222,102
  Match rate: 94.61%


In [4]:

# --- Inspect missing ---
# Some missing is expected (delisted stocks, data gaps)
# But if match rate < 80% something is wrong with date alignment
if kelly_with_ret['ret'].notna().mean() < 0.80:
    print("\nWARNING: Match rate below 80% — checking date alignment...")
    
    # Sample some missing rows to diagnose
    missing = kelly_with_ret[kelly_with_ret['ret'].isna()]
    print("\nSample missing rows:")
    print(missing[['permno','date']].head(10))
    
    # Check if those permnos exist in CRSP at all
    sample_permnos = missing['permno'].head(5).values
    for p in sample_permnos:
        crsp_dates = crsp[crsp['permno'] == p]['date']
        kelly_dates = kelly_with_ret[kelly_with_ret['permno'] == p]['date']
        print(f"\npermno {p}:")
        print(f"  CRSP dates:  {crsp_dates.min()} to {crsp_dates.max()}")
        print(f"  Kelly dates: {kelly_dates.min()} to {kelly_dates.max()}")
else:
    print("\nMatch rate looks good.")



Match rate looks good.


In [5]:

# --- Save ---
kelly_with_ret.to_csv('kelly_with_returns.csv', index=False)
print("\nSaved: kelly_with_returns.csv")
print(f"Columns: {kelly_with_ret.columns.tolist()}")


Saved: kelly_with_returns.csv
Columns: ['permno', 'DATE', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade', 'sic2', 'date', 're

In [6]:
# Check 1: Where are the missing returns concentrated?
missing = kelly_with_ret[kelly_with_ret['ret'].isna()]

print("Missing by decade:")
missing['year'] = missing['date'].dt.year
print(missing.groupby((missing['year']//10)*10).size())

print("\nMissing by exchange:")
# If exchcd came through from CRSP check it
# Otherwise just check date distribution
print(missing['date'].dt.year.value_counts().sort_index().head(20))

# Check 2: Spot check a few matched rows to confirm values look right
print("\nSample of matched rows (ret column):")
matched = kelly_with_ret[kelly_with_ret['ret'].notna()]
print(matched[['permno','date','ret']].sample(10, random_state=42))

print("\nReturn distribution in merged dataset:")
print(kelly_with_ret['ret'].describe())

# Check 3: Confirm date range looks right
print("\nDate range of matched rows:")
print(matched['date'].min(), "to", matched['date'].max())

Missing by decade:
year
1950       84
1960     2440
1970    11399
1980    72099
1990    63235
2000    42661
2010    25504
2020     4680
dtype: int64

Missing by exchange:


C:\Users\lusip\AppData\Local\Temp\ipykernel_1712\2467825674.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing['year'] = missing['date'].dt.year


date
1957      34
1958      27
1959      23
1960      35
1961      34
1962     307
1963     538
1964     437
1965     352
1966     247
1967     206
1968     163
1969     121
1970      87
1971      70
1972      86
1973    1011
1974    1672
1975    1631
1976    2092
Name: count, dtype: int64

Sample of matched rows (ret column):
         permno       date       ret
970415    43617 1983-03-01 -0.102273
398567    28952 1973-09-01 -0.032787
814741    59520 1980-09-01 -0.103586
1692884   11235 1992-05-01  0.114286
851011    15588 1981-05-01  0.075145
947816    32600 1982-11-01  0.250000
1147450   43481 1985-08-01 -0.047170
2119167   76424 1996-11-01  0.090909
3989334   45306 2020-04-01 -0.048468
1971539   11913 1995-06-01  0.320513

Return distribution in merged dataset:
count    3.895198e+06
mean     8.214808e-03
std      1.359115e-01
min     -7.248406e-01
25%     -5.793700e-02
50%      4.340000e-04
75%      6.439400e-02
max      1.723182e+00
Name: ret, dtype: float64

Date range of matched

In [8]:
# Don't use .copy() — write directly to csv in chunks
# This avoids loading the full dataset into memory twice

output_file = 'kelly_final.csv'
chunk_size = 500000  # 500k rows at a time

# Get column list minus DATE
cols_to_keep = [c for c in kelly_with_ret.columns if c != 'DATE']

# Filter to non-missing ret rows only, write in chunks
first_chunk = True
total_written = 0

for start in range(0, len(kelly_with_ret), chunk_size):
    chunk = kelly_with_ret.iloc[start:start+chunk_size]
    
    # Filter missing ret
    chunk = chunk[chunk['ret'].notna()]
    chunk = chunk[cols_to_keep]
    
    # Write header only on first chunk
    chunk.to_csv(output_file, 
                 mode='w' if first_chunk else 'a',
                 header=first_chunk,
                 index=False)
    
    total_written += len(chunk)
    first_chunk = False
    print(f"Written {total_written:,} rows...")

print(f"\nDone. Total rows written: {total_written:,}")
print("Saved: kelly_final.csv")

# Free memory
del kelly_with_ret
del kelly
del crsp

import gc
gc.collect()
print("Memory freed.")

Written 494,043 rows...
Written 970,620 rows...
Written 1,414,466 rows...
Written 1,869,438 rows...
Written 2,349,629 rows...
Written 2,822,100 rows...
Written 3,299,797 rows...
Written 3,780,727 rows...
Written 3,895,198 rows...

Done. Total rows written: 3,895,198
Saved: kelly_final.csv
Memory freed.


In [1]:
import pandas as pd

# Quick verification without loading full file
sample = pd.read_csv('kelly_final.csv', nrows=5)
print("Columns:", sample.columns.tolist())
print("Sample:\n", sample[['permno','date','ret']].head())

# Count rows
row_count = sum(1 for _ in open('kelly_final.csv')) - 1
print(f"\nTotal rows: {row_count:,}")

# Check date range using chunked read
dates = []
for chunk in pd.read_csv('kelly_final.csv', 
                          usecols=['date'], 
                          chunksize=500000):
    dates.append(chunk['date'].min())
    dates.append(chunk['date'].max())

print(f"Date range: {min(dates)} to {max(dates)}")

Columns: ['permno', 'mvel1', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'bm', 'bm_ia', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof', 'orgcap', 'pchcapx_ia', 'pchcurrat', 'pchdepr', 'pchgm_pchsale', 'pchquick', 'pchsale_pchinvt', 'pchsale_pchrect', 'pchsale_pchxsga', 'pchsaleinv', 'pctacc', 'ps', 'quick', 'rd', 'rd_mve', 'rd_sale', 'realestate', 'roic', 'salecash', 'saleinv', 'salerec', 'secured', 'securedind', 'sgr', 'sin', 'sp', 'tang', 'tb', 'aeavol', 'cash', 'chtx', 'cinvest', 'ear', 'nincr', 'roaq', 'roavol', 'roeq', 'rsup', 'stdacc', 'stdcf', 'ms', 'baspread', 'ill', 'maxret', 'retvol', 'std_dolvol', 'std_turn', 'zerotrade', 'sic2', 'date', 'ret', 'ret_raw', 'me']
Sample:
    permno